# Assignment 2: Text Preprocessing and Classification

This notebook demonstrates **text preprocessing**, **feature engineering**, **text classification**, and **model evaluation** on a labeled text dataset.

## Dataset
We use the **20 Newsgroups** dataset from `scikit-learn`, a standard labeled text classification dataset.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

In [ ]:
# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

In [ ]:
# Load dataset
data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
texts = data.data
labels = data.target

print('Number of documents:', len(texts))

## Text Preprocessing
Steps performed:
- Tokenization
- Stopword Removal
- Stemming
- Lemmatization
- POS Tagging

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    stemmed = [stemmer.stem(t) for t in tokens]
    lemmatized = [lemmatizer.lemmatize(t) for t in tokens]
    pos_tags = pos_tag(tokens)
    return ' '.join(lemmatized), stemmed, pos_tags

In [ ]:
processed_texts = []
for text in texts:
    clean_text, _, _ = preprocess_text(text)
    processed_texts.append(clean_text)

## Feature Engineering
We use the following representations:
- Bag of Words (Unigram, Bigram)
- TF-IDF Vectors

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    processed_texts, labels, test_size=0.2, random_state=42)

# TF-IDF Vectorization with unigrams and bigrams
tfidf = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Text Classification
We train a **Multinomial Naive Bayes** classifier using TF-IDF features.

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

## Model Evaluation
Evaluation metrics used:
- Accuracy
- Precision
- Recall
- F1-score

In [ ]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted')

print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)

## Performance Report
The model achieves competitive performance on the 20 Newsgroups dataset.
TF-IDF with bigrams improves contextual understanding compared to simple Bag-of-Words.